# DSAT Difficulty Classification Pipeline

Automated system for classifying Digital SAT (DSAT) question difficulty using computer vision and OCR.

## Overview
This notebook processes official DSAT practice test PDFs to extract difficulty labels from answer keys, enabling automated organization of questions by difficulty level.

**Note:** By default, this notebook uses cached features for reproducibility. See `data/README.md` for instructions on running full OCR pipeline.

## Configuration

In [ ]:
import random
import numpy as np
from pathlib import Path

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Runtime configuration
RUN_OCR = False  # Set to True to run full OCR pipeline (requires raw PDFs)
USE_CACHED_FEATURES = True  # Use pre-extracted features for reproducibility
DEBUG_MODE = False  # Enable detailed logging

# Paths (relative to repo root)
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = REPO_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
DERIVED_DIR = DATA_DIR / "derived"
OUTPUT_DIR = REPO_ROOT / "output"

# Create necessary directories
for dir_path in [DERIVED_DIR, OUTPUT_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print(f"✓ Configuration loaded")
print(f"  - Repository root: {REPO_ROOT}")
print(f"  - OCR mode: {'Enabled' if RUN_OCR else 'Disabled (using cached features)'}")
print(f"  - Random seed: {RANDOM_SEED}")

## Package Installation

**Note:** OCR dependencies are only needed if `RUN_OCR=True`. Skip this cell if using cached features.

In [ ]:
# Only run if OCR is enabled
if RUN_OCR:
    !pip install -q pdfminer.six pypdf pdf2image opencv-python-headless
    !apt-get install -y poppler-utils
    print("✓ OCR dependencies installed")
else:
    print("⊘ Skipping OCR dependencies (RUN_OCR=False)")

## Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Core libraries (always needed)
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# OCR libraries (conditional)
if RUN_OCR:
    import cv2
    from pdf2image import convert_from_path
    from pypdf import PdfReader, PdfWriter
    import re

print("✓ Libraries imported successfully")

## OCR Runtime Note

**Full OCR over all question images is disabled by default** for reproducibility and runtime efficiency. 

The repository includes cached derived features in `data/derived/`. To reproduce OCR feature extraction:
1. Follow instructions in `data/README.md` to obtain official DSAT PDFs
2. Set `RUN_OCR=True` in the configuration cell above
3. Rerun all cells

## Core Functions

In [ ]:
def detect_difficulty_cv(key_pdf_path, dpi=200, debug_every=0, comprehensive_debug=False):
    """
    Detect difficulty labels from answer key PDF using computer vision.
    
    Parameters:
    -----------
    key_pdf_path : Path or str
        Path to the answer key PDF
    dpi : int
        Resolution for PDF conversion (default: 200)
    debug_every : int
        Print debug info every N pages (0 = off)
    comprehensive_debug : bool
        Enable detailed debugging output
    
    Returns:
    --------
    list of str : Difficulty labels for each page
    """
    if not RUN_OCR:
        raise RuntimeError("OCR is disabled. Set RUN_OCR=True to use this function.")
    
    # Crop boundaries (fractions of page)
    crop_params = {'top': 0.12, 'bottom': 0.148, 'left': 0.77, 'right': 0.86}
    
    # Convert PDF to images
    pages = convert_from_path(str(key_pdf_path), dpi=dpi)
    print(f"Converted {len(pages)} pages from PDF")
    
    results = []
    stats = {'boxes': [], 'confidences': [], 'failed': []}
    
    for page_num, page in enumerate(pages, start=1):
        # Convert to grayscale
        img = np.array(page.convert("L"))
        h, w = img.shape
        
        # Apply crop
        r0 = int(crop_params['top'] * h)
        r1 = int(crop_params['bottom'] * h)
        c0 = int(crop_params['left'] * w)
        c1 = int(crop_params['right'] * w)
        crop = img[r0:r1, c0:c1]
        
        # Binarize
        _, binary = cv2.threshold(crop, 127, 255, cv2.THRESH_BINARY_INV)
        
        # Find contours
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        # Filter valid boxes (size heuristics)
        valid_boxes = []
        for cnt in contours:
            x, y, bw, bh = cv2.boundingRect(cnt)
            area = bw * bh
            if 50 < area < 5000 and 0.2 < bw/max(bh, 1) < 5:
                valid_boxes.append((x, y, bw, bh))
        
        stats['boxes'].append(len(valid_boxes))
        
        # Determine difficulty based on box count
        if len(valid_boxes) >= 3:
            difficulty = "Hard"
            confidence = min(0.99, 0.85 + len(valid_boxes) * 0.05)
        elif len(valid_boxes) == 2:
            difficulty = "Medium"
            confidence = 0.90
        elif len(valid_boxes) == 1:
            # Could be Easy or ambiguous
            difficulty = "Easy" if bw > 15 else "Unknown"
            confidence = 0.75
        else:
            difficulty = "Unknown"
            confidence = 0.0
            stats['failed'].append(page_num)
        
        stats['confidences'].append(confidence)
        results.append(difficulty)
        
        if debug_every > 0 and page_num % debug_every == 0:
            print(f"  Page {page_num}: {difficulty} ({len(valid_boxes)} boxes, conf={confidence:.2f})")
    
    # Summary
    print(f"\n=== DETECTION SUMMARY ===")
    print(f"Total pages processed: {len(pages)}")
    print(f"Results distribution: {dict(Counter(results))}")
    print(f"Average boxes found per page: {np.mean(stats['boxes']):.1f}")
    print(f"Average confidence: {np.mean(stats['confidences']):.2f}")
    print(f"Failed detections: {len(stats['failed'])}")
    if stats['failed']:
        print(f"Failed pages: {stats['failed']}")
    
    return results

print("✓ Core functions defined")

## Data Loading

Load cached features or run OCR pipeline based on configuration.

In [ ]:
if USE_CACHED_FEATURES and not RUN_OCR:
    # Load cached features
    features_path = DERIVED_DIR / "difficulty_labels.csv"
    
    if features_path.exists():
        df = pd.read_csv(features_path)
        print(f"✓ Loaded cached features from {features_path}")
        print(f"  Shape: {df.shape}")
        print(f"  Columns: {list(df.columns)}")
    else:
        print(f"⚠ No cached features found at {features_path}")
        print(f"  To create features:")
        print(f"  1. Place PDFs in {RAW_DIR}")
        print(f"  2. Set RUN_OCR=True")
        print(f"  3. Rerun all cells")
        df = None

elif RUN_OCR:
    # Run OCR pipeline
    print("Running OCR pipeline...")
    
    # Check if raw data exists
    if not RAW_DIR.exists() or not any(RAW_DIR.glob("*.pdf")):
        print(f"⚠ No PDF files found in {RAW_DIR}")
        print(f"  See data/README.md for instructions")
        df = None
    else:
        # Process PDFs
        all_results = []
        
        for key_pdf in sorted(RAW_DIR.glob("*_Key.pdf")):
            print(f"\nProcessing {key_pdf.name}...")
            labels = detect_difficulty_cv(key_pdf, debug_every=50)
            
            # Create records
            test_name = key_pdf.stem.replace("_Key", "")
            for page_num, label in enumerate(labels, start=1):
                all_results.append({
                    'test': test_name,
                    'page': page_num,
                    'difficulty': label
                })
        
        df = pd.DataFrame(all_results)
        
        # Cache results
        cache_path = DERIVED_DIR / "difficulty_labels.csv"
        df.to_csv(cache_path, index=False)
        print(f"\n✓ Cached features to {cache_path}")

else:
    print("⚠ Invalid configuration: Set either USE_CACHED_FEATURES=True or RUN_OCR=True")
    df = None

## Exploratory Data Analysis

In [ ]:
if df is not None:
    # Overall distribution
    print("=== Difficulty Distribution ===")
    print(df['difficulty'].value_counts())
    print(f"\nTotal pages: {len(df)}")
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Overall distribution
    df['difficulty'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
    axes[0].set_title('Overall Difficulty Distribution')
    axes[0].set_xlabel('Difficulty Level')
    axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=0)
    
    # By test
    pivot = df.pivot_table(index='test', columns='difficulty', values='page', aggfunc='count', fill_value=0)
    pivot.plot(kind='bar', ax=axes[1], stacked=True)
    axes[1].set_title('Difficulty Distribution by Test')
    axes[1].set_xlabel('Test')
    axes[1].set_ylabel('Page Count')
    axes[1].legend(title='Difficulty')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig(REPO_ROOT / 'images' / 'difficulty_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n✓ Analysis complete")
else:
    print("⊘ No data available for analysis")

## Summary Statistics

In [ ]:
if df is not None:
    print("=== Summary Statistics ===")
    print(f"\nUnique tests: {df['test'].nunique()}")
    print(f"Total pages processed: {len(df)}")
    print(f"\nDifficulty breakdown:")
    for difficulty, count in df['difficulty'].value_counts().items():
        pct = 100 * count / len(df)
        print(f"  {difficulty:10s}: {count:3d} ({pct:5.1f}%)")
    
    unknown_count = (df['difficulty'] == 'Unknown').sum()
    quality = 100 * (1 - unknown_count / len(df))
    print(f"\nDetection quality: {quality:.1f}% (labeled as non-Unknown)")
else:
    print("⊘ No data available for summary")

## Conclusion

This pipeline demonstrates:
- Computer vision-based difficulty detection from answer key PDFs
- Reproducible workflow using cached features
- Automated organization of DSAT questions by difficulty level

### Next Steps
- Improve detection accuracy for edge cases
- Add text-based feature extraction for enhanced classification
- Build supervised ML model using extracted features
- Create Streamlit dashboard for interactive exploration